In [3]:
import random
import rich
import sys
import json
sys.path.append("../../")
from sotopia.database import AgentProfile, EpisodeLog, EnvironmentProfile

In [4]:
# Check the store tag for episodes

all_epi_pks = list(EpisodeLog.all_pks())
epi_store_tag = []
for pk in all_epi_pks:
    epi = EpisodeLog.get(pk)
    if epi.tag not in epi_store_tag:
        epi_store_tag.append(epi.tag)
epi_store_tag

['test_finetune_8b_v3',
 'llama3.18b_lora',
 'whole',
 'llama3_8b_lora_finetune_filtered',
 'test_finetune_8b',
 'taskeval_fewshot',
 'test_finetune_8b_v2',
 'llama3_8b_finetune_full',
 'taskeval_fewshot_plausible_v2',
 'coop_with_flausible_move_v3',
 'new_taskeval_gpt_4_101',
 'llama3_8B_England-German',
 'te_without_previous_llama3_70b',
 'taskeval',
 'test_demo_v2',
 'coop_with_actual_move_thres_0.02',
 'random_sample_100_games',
 'specific_human_anno_llama3_70b',
 'llama3_8b_lora_finetune_v3',
 'new_taskeval_llama3_70b',
 'ntaske',
 'test_finetune_8b_v1',
 'specific_human_anno_gpt_4_new',
 'tv3',
 'demo_sample_gpt_with_actual',
 'new_taskeval_llama3_within_10_turns',
 'demo_gpt_4o',
 'taskeval_fewshot_plausible_parse',
 'te_n_with_previous_llama3_70b',
 'test_demo_sample_gpt_with_actual',
 'demo_finetune_effect_test',
 'demo_llama',
 'taskeval_fewshot_plausible',
 'test_demo_v4',
 'test_llama3.1',
 'specific_human_anno_gpt_4',
 'test_demo',
 'llama3-70b-analysis',
 'coop_with_actua

In [5]:
epis = []
all_epi_pks = list(EpisodeLog.all_pks())
for pk in all_epi_pks:
    epi = EpisodeLog.get(pk)
    if epi.tag == 'llama3.170b':
    # if epi.tag == 'new_taskeval_llama3_70b':
        epis.append(epi)
len(epis)

0

In [2]:
index = 2
epi = epis[index]
rich.print(epi.messages)

NameError: name 'epis' is not defined

## Read The Formatted Episode

In [6]:
import sys
import os

sys.path.append("../")
from episode_utils import (
    process_conversation, 
    format_diplomacy_data, 
    get_game_phase_env_from_episode,
    replace_names_with_countries,
    process_conversation_to_intent
)

def get_episodes(tag):
    all_task_pks = list(EpisodeLog.all_pks())
    episodelogs = []
    for pk in all_task_pks:
        episo = EpisodeLog.get(pk)
        if episo.tag == tag:
            # pdb.set_trace()
            env = get_game_phase_env_from_episode(episo)
            episodelogs.append({"game_id": env.game_id, "agents": episo.agents, "phase_name": env.phase_name, "env_uuid": env.pk, "env": env, "episode": episo})
    return episodelogs

def format_episode(episodes):
    new_episodes = []
    agent_profiles = []
    all_character_pks = list(AgentProfile.all_pks())
    for pk in all_character_pks:
        agent_profiles.append(AgentProfile.get(pk))
    for episode in episodes:
        # pdb.set_trace()
        episode['dialogue'] = replace_names_with_countries(process_conversation(episode["episode"].messages), agent_profiles)
        episode['intent_dialogue'] = process_conversation_to_intent(episode['dialogue'])
        episode['unit_center'] = format_diplomacy_data(episode["env"].scenario)
        episode['reasoning']  = episode['episode'].reasoning
        agents = episode['env'].agent_powers
        rewards = episode['episode'].rewards
        rewards_info = ""
        for i in range(len(agents)):
            rewards_info += f"{agents[i]}: {rewards[i][1]}\n"
        episode['rewards'] = rewards_info
        new_episodes.append(episode)
    return new_episodes


In [7]:
formatted_episodes = format_episode(get_episodes('test_llama3.1_v2'))

In [8]:
rich.print(formatted_episodes[0]['intent_dialogue'])
rich.print(formatted_episodes[0]['reasoning'])
rich.print(formatted_episodes[0]['rewards'])

0 FRANCE -> GERMANY: Alors, Germany, c'est temps de faire des projets ensemble, n'est-ce pas? Nous pourrions 
discuter de la manière dont nous pouvons travailler ensemble pour contrôler les centres de suppleance.
1 GERMANY -> FRANCE: Bien sür, France. Je suis heureux de discuter avec vous. Mais avant de faire des projets 
ensemble, je pense qu'il serait utile de parler des mouvements d'armées que nous pouvons faire ce tour pour notre 
profit. Puis-je savoir si vous avez des intentions de faire quelque chose dans le nord ou si vous pensez que nous 
pourrions travailler ensemble pour contrôler Mun.
2 FRANCE -> GERMANY: Ah, bien sûr, Germany. Je vois que vous avez déjà des pensées sur la manière de travailler 
ensemble. Mais avant de faire quoi que ce soit, je pense qu'il est essentiel que nous assurions que nous sommes sur
la même longueur d'onde quant aux intentions de l'Italie. Qu'en pensez-vous ?
3 GERMANY -> FRANCE: Je comprends vos inquiétudes sur la position italienne, mais je pense que nous devrions nous 
concentrer sur la situation actuelle et discuter de la manière dont nous pouvons travailler ensemble pour protéger 
nos intérêts. Puis-je savoir si vous avez pensé à la possibilité de soutenir la France contre l'Italie sur l'Ionie 
ce tour?
4 FRANCE -> GERMANY: Je comprends votre point de vue, Germany, mais je pense que nous devrions garder un œil sur 
l'Italie. Si elle s'approche de l'Ionie, nous serons dans une position vulnérable. Faut-il envisager de soutenir la
Russie pour faire pression sur l'Italie ?
5 GERMANY -> FRANCE: Je comprends vos inquiétudes sur la situation, mais je pense que nous devrions nous concentrer
sur la manière dont nous pouvons travailler ensemble pour protéger nos intérêts. Pourquoi ne pas soutenir la France
contre l'Italie sur l'Ionie ce tour ? Cela pourrait être bénéfique pour nous deux.
6 FRANCE -> GERMANY: Je vois que vous avez pensé à soutenir la France contre l'Italie sur l'Ionie ce tour, Germany.
C'est une bonne idée, mais qu'en pensez-vous de soutenir la Russie pour faire pression sur l'Italie ?
7 GERMANY -> FRANCE: Je suis d'accord avec vous, France. Soutenir la Russie pour faire pression sur l'Italie 
pourrait être une excellente idée. Mais avant de faire quoi que ce soit, je pense qu'il est essentiel que nous 
assurions que nous sommes sur la même longueur d'onde quant aux intentions de l'Italie. Qu'en pensez-vous ?
8 FRANCE -> GERMANY: Je suis dâcidié, Germany, que nous devrions soutenir la Russie pour faire pression sur 
l’Italie. Cela nous donnerait une position de force pour négocier avec elle. Qu'en pensez-vous ?
9 GERMANY -> FRANCE: Je suis d'accord avec l'idée de soutenir la Russie pour faire pression sur l'Italie. Mais 
avant de faire quoi que ce soit, je pense qu'il est essentiel que nous assurions que nous sommes sur la même 
longueur d'onde quant aux intentions de l'Italie. Qu'en pensez-vous ?
10 FRANCE -> GERMANY: Je vois que nous sommes d'accord sur la nécessité de faire pression sur l'Italie. Mais qu'en 
pensez-vous de soutenir la Russie pour contrôler le centre de Moscou ? Cela pourrait nous donner une position de 
force pour négocier avec l'Italie et la Russie.
11 GERMANY -> FRANCE: Je pense que nous devrions soutenir la Russie pour contrôler le centre de Moscou, en effet, 
cela nous donnerait une position de force pour négocier avec l'Italie et la Russie.
12 FRANCE -> GERMANY: Je suis en accord avec vous, Germany, pour soutenir la Russie pour contrôler le centre de 
Moscou. Cela nous donnerait une position de force pour négocier avec l'Italie et la Russie. Qu'en pensez-vous de la
manière dont nous pouvons travailler ensemble pour protéger nos intérêts ?

Environment comments: terminated: The conversation is too long; 
Agent 1 comments:
ethos: <ethos> Sam effectively shares his strategic intentions regarding the situation with Italy and the potential
for collaboration with Germany. He proposes actionable steps, such as supporting Russia to pressure Italy, which 
demonstrates a clear understanding of the game dynamics. However, he could have provided more specific details 
about his own plans to strengthen his credibility.
logos: <logos> Sam's reasoning is logical as he connects the need to pressure Italy with the potential benefits of 
supporting Russia. He reflects on the implications of Italy's movements and suggests a collaborative approach, 
which shows a coherent thought process. However, he could enhance his argument by providing more detailed 
justifications for his proposed actions.
pathos: <pathos> Sam maintains a friendly tone throughout the dialogue, showing openness to collaboration and 
valuing Germany's input. His use of inclusive language fosters a sense of partnership. However, he could 
incorporate more emotional appeals or reassurances to strengthen the rapport further.
Agent 2 comments:
ethos: <ethos> James establishes his credentials by acknowledging Sam's concerns and expressing a willingness to 
collaborate. He provides relevant information about his own intentions, such as holding Munich, which adds 
credibility. However, he could have been more specific about his plans to enhance his ethos.
logos: <logos> James demonstrates logical reasoning by discussing the need to focus on the current situation and 
suggesting specific actions, such as supporting France against Italy. His thought process is coherent, but he could
improve by providing more detailed justifications for his proposed moves.
pathos: <pathos> James maintains a friendly demeanor and shows a willingness to engage with Sam's ideas. His 
responses are supportive, which helps build rapport. However, he could enhance his emotional appeal by 
incorporating humor or more personal touches to make the interaction feel warmer.

France: {'ethos': 7.0, 'logos': 8.0, 'pathos': 7.0, 'overall_score': 7.333333333333333}
Germany: {'ethos': 6.0, 'logos': 7.0, 'pathos': 6.0, 'overall_score': 6.333333333333333}

In [ ]:
# # Context Length Calculation
# from transformers import AutoTokenizer
# model_path = "/data/models/huggingface/meta-llama/Meta-Llama-3-8B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_path)